# 02 - Camadas estáticas

| | |
|---|---|
| **Tarefa do plano** | S1-05 - DEM, HAND, WorldCover, JRC GSW, HydroRIVERS (pasta `static/` completa) |
| **Variáveis (Parte I)** | #3 rede hidrográfica, #33 elevação, #34 declive, #35 HAND, #36 distância ao rio, #37 acumulação de fluxo, #39 aspeto, #40 WorldCover, #41 ocorrência de água JRC |
| **Fonte** | Copernicus DEM GLO-30, MERIT Hydro, ESA WorldCover v200, JRC GSW v1.4, HydroSHEDS - via Earth Engine |
| **Saída** | `data/raw/static/` |

Todos os rasters são descarregados na grelha do projeto, por isso já ficam alinhados.

Correr primeiro `00_setup_and_authentication` e `01_study_area_aoi`. Todos os
downloads podem ser retomados: os ficheiros que já existem são saltados, e cada
ficheiro fica registado em `data/metadata/download_manifest.csv`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import ee
import pandas as pd
import matplotlib.pyplot as plt

from ete import config, datasets, gee, series
from ete import grid as G

cfg = config.load_config()
gee.init(cfg)

aoi = G.load_aoi(cfg)                       # AOI das imagens (EPSG:4326)
region = gee.ee_geometry(aoi)
grid30 = G.load_grid(cfg)                   # grelha do projeto a 30 m (EPSG:32736)
grid10 = G.load_grid(cfg, 10)               # grelha a 10 m, encaixada na de 30 m
climate_bounds = tuple(G.load_aoi(cfg, "aoi_climate").total_bounds)   # O, S, E, N
MASK = gee.ee_geometry(G.imagery_mask(cfg))  # distrito + margem: fora daqui as imagens mensais ficam sem dados

s = cfg["datasets"]["static"]

## 1. Terreno (30 m)

O declive e o aspeto são calculados pelo Earth Engine sobre o DEM nativo. O MERIT
Hydro é um produto de 90 m: o HAND e a área a montante são reamostrados (vizinho
mais próximo) para 30 m.

In [ ]:
dem_col = ee.ImageCollection(s["dem"]).select("DEM")
dem = dem_col.mosaic().setDefaultProjection(dem_col.first().projection())
terrain = (dem.rename("elevation")
           .addBands(ee.Terrain.slope(dem).rename("slope"))
           .addBands(ee.Terrain.aspect(dem).rename("aspect")))
gee.download_static(cfg, "dem", s["dem"], terrain, ["elevation"], grid30, "static_terrain")
gee.download_static(cfg, "slope", s["dem"], terrain, ["slope"], grid30, "static_terrain")
gee.download_static(cfg, "aspect", s["dem"], terrain, ["aspect"], grid30, "static_terrain")

merit = ee.Image(s["merit_hydro"]).select(["hnd", "upa"], ["hand", "upstream_area"])
hand_file = gee.download_static(cfg, "hand", s["merit_hydro"], merit, ["hand"], grid30, "static_terrain")
gee.download_static(cfg, "flow_accumulation", s["merit_hydro"], merit, ["upstream_area"], grid30, "static_terrain")

## 2. Cobertura do solo (10 m) e histórico de água superficial (30 m)

In [ ]:
worldcover = ee.ImageCollection(s["worldcover"]).first().select(["Map"], ["land_cover"])
gee.download_static(cfg, "worldcover", s["worldcover"], worldcover, ["land_cover"], grid10, "static_land_cover", dtype="uint8")

jrc_bands = ["occurrence", "seasonality", "recurrence", "transition", "max_extent"]
jrc_file = gee.download_static(cfg, "jrc_gsw", s["jrc_gsw"], ee.Image(s["jrc_gsw"]), jrc_bands, grid30, "static_surface_water", dtype="uint8")

## 3. Rede hidrográfica (vetor) e distância ao rio

Os rios são obtidos para toda a AOI climática. O raster de distância é calculado
localmente numa grelha alargada em 10 km, para que os rios logo fora da AOI
também contem; é uma camada derivada e fica em `data/interim/static/`.

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
from shapely.geometry import box

rivers_file = config.path(cfg, "static_hydrography", "rivers_incomati.geojson")
if not rivers_file.exists():
    rivers_fc = ee.FeatureCollection(s["rivers"]).filterBounds(ee.Geometry.Rectangle(list(climate_bounds)))
    gee.download_vector(rivers_fc, rivers_file)
rivers = gpd.read_file(rivers_file).to_crs(grid30.crs)
print(len(rivers), "troços de rio")

pad = round(10_000 / grid30.res)
padded = G.Grid(grid30.crs, grid30.res, grid30.x0 - pad * grid30.res, grid30.y0 + pad * grid30.res,
                grid30.width + 2 * pad, grid30.height + 2 * pad)
near = rivers[rivers.intersects(box(*padded.bounds))]
if near.empty:
    raise ValueError("nenhum troço de rio a menos de 10 km da AOI das imagens - verificar a camada de rios")
river_px = rasterize(near.geometry, out_shape=(padded.height, padded.width), transform=padded.transform, all_touched=True)
distance = (distance_transform_edt(river_px == 0) * grid30.res)[pad:-pad, pad:-pad].astype("float32")

dist_file = config.path(cfg, "interim", "static", f"distance_to_river_{cfg['project']['aoi_name']}_30m.tif")
with rasterio.open(hand_file) as src:
    profile = src.profile
profile.update(count=1, dtype="float32", nodata=-9999.0)
with rasterio.open(dist_file, "w", **profile) as dst:
    dst.write(distance, 1)
    dst.set_band_description(1, "distance_to_river_m")
print("escrito", dist_file.relative_to(ROOT))

## 4. Controlo de qualidade

In [ ]:
terrain_dir = config.path(cfg, "static_terrain")
name = cfg["project"]["aoi_name"]
fig, axes = plt.subplots(2, 3, figsize=(16, 12))
series.show_raster(terrain_dir / f"dem_{name}_30m.tif", ax=axes[0, 0], cmap="terrain")
series.show_raster(terrain_dir / f"slope_{name}_30m.tif", ax=axes[0, 1], cmap="magma", vmax=15)
series.show_raster(hand_file, ax=axes[0, 2], cmap="viridis_r", vmax=30)
series.show_raster(jrc_file, band="occurrence", ax=axes[1, 0], cmap="Blues")
series.show_raster(config.path(cfg, "static_land_cover") / f"worldcover_{name}_10m.tif", ax=axes[1, 1], cmap="tab20")
series.show_raster(dist_file, ax=axes[1, 2], cmap="cividis")
for ax in axes.flat:
    near.plot(ax=ax, color="red", linewidth=0.6)
    ax.set_xlim(grid30.bounds[0], grid30.bounds[2]); ax.set_ylim(grid30.bounds[1], grid30.bounds[3])
plt.tight_layout()

Verificar se a linha vermelha do rio segue o corredor de HAND baixo e a água do
JRC. A linha de rio do HydroSHEDS tem uma exatidão de algumas centenas de metros;
se estiver visivelmente fora do canal real, a distância ao rio deve mais tarde
ser recalculada a partir da água permanente do JRC (semana 2).